### 1. Install vLLM server

vLLM exposes an OpenAI-compatible local HTTP API, so the notebook can use the standard `openai` Python client instead of Ollama's `/api/generate` endpoint.

> Windows note: vLLM is Linux-first. On Windows, run it inside WSL2 / Ubuntu, or on a Linux VM/server with NVIDIA CUDA available.

Install packages in the environment where the vLLM server will run:

```bash
pip install vllm openai requests
```

Start the vLLM server in a separate terminal before running the notebook:

```bash
vllm serve Qwen/Qwen3.5-0.8B \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key EMPTY \
  --dtype auto \
  --gpu-memory-utilization 0.85
```

If you run vLLM on the same machine as Jupyter, the notebook will connect to:

```text
http://localhost:8000/v1
```

If vLLM runs on another machine or VM, change `VLLM_BASE_URL` in the setup cell, for example:

```python
#VLLM_BASE_URL = "http://192.168.50.131:8000/v1"
```

### 2. Model choice

Default model in this notebook:

```text
Qwen/Qwen2.5-3B-Instruct
```

For a 12 GB GPU, this is a safer starting point than a bigger 7B/8B model. You can change the model by editing `MODEL_NAME`, but the value should match the model served by `vllm serve`.


In [ ]:
!pip install vllm openai requests

### 02-ai-workflows

In [ ]:
#01-first-workflow-vllm-openai-compatible-api

import os
from openai import OpenAI

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")
MODEL_NAME = os.getenv("VLLM_MODEL", "Qwen/Qwen3.5-0.8B")

client = OpenAI(base_url=VLLM_BASE_URL,api_key=VLLM_API_KEY,timeout=120.0,)


def call_vllm(prompt: str,system_prompt: str = "You are a helpful assistant.",max_tokens: int = 500,temperature: float = 0.7) -> str:

    response = client.chat.completions.create(
                                                model=MODEL_NAME,
                                                messages=[{"role": "system", "content": system_prompt},{"role": "user", "content": prompt.strip()}],
                                                temperature=temperature,
                                                max_tokens=max_tokens,
                                            )

    return response.choices[0].message.content.strip()


def generate_x_post(topic: str) -> str:
    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Topic:
                {topic}
            """

    return call_vllm(prompt, max_tokens=350, temperature=0.7)


def main():
    usr_input = input("What should the post be about? ")
    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)


if __name__ == "__main__":
    main()

### 02-using-vLLM-via-openai-sdk


In [ ]:
def generate_x_post(topic: str) -> str:
    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Here's the topic provided by the user for which you need to generate a post:
                <topic>
                {topic}
                </topic>
            """

    return call_vllm(prompt, max_tokens=350, temperature=0.7)


def main():
    usr_input = input("What should the post be about? ")
    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)


if __name__ == "__main__":
    main()

### 03-few-shot-prompting

In [ ]:
import json


def generate_x_post(topic: str) -> str:
    with open("post-examples.json", "r", encoding="utf-8") as f:
        examples = json.load(f)

    examples_str = ""
    for i, example in enumerate(examples, 1):
        examples_str += f"""
                            <example-{i}>
                                    <topic>
                                    {example['topic']}
                                    </topic>

                                    <generated-post>
                                    {example['post']}
                                    </generated-post>
                            </example-{i}>
                        """

    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Here's the topic provided by the user for which you need to generate a post:
                <topic>
                {topic}
                </topic>

                Here are some examples of topics and generated posts:
                <examples>
                    {examples_str}
                </examples>

                Please use the tone, language, structure , and style of the examples provided above to generate a post that is engaging and relevant to the topic provided by the user.
                Don't use the content from the examples!
                """

    return call_vllm(prompt, max_tokens=350, temperature=0.7)


def main():
    usr_input = input("What should the post be about? ")
    x_post = generate_x_post(usr_input)
    print("Generated X post")
    print(x_post)


if __name__ == "__main__":
    main()

### 04-multi-step-multi-model

In [ ]:
import json
import requests


def get_website_html(url: str) -> str:
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # Raise an error for bad responses
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching the URL {url}: {e}")
        return ""


def extract_core_website_content(html: str) -> str:
    prompt=f"""
                You are an expert web content extractor. Your task is to extract the core content from a given HTML page.
                The core content should be the main text, excluding navigation, footers, and other non-essential elements like scripts etc.

                Here is the HTML content:
                <html>
                {html}
                </html>

                Please extract the core content and return it as plain text.
            """

    return call_vllm(prompt, max_tokens=1200, temperature=0.2)


def summarize_content(content: str) -> str:

    prompt=f"""
                You are an expert summarizer. Your task is to summarize the provided content into a concise and clear summary.

                Here is the content to summarize:
                <content>
                {content}
                </content>

                Please provide a brief summary of the main points in the content. Prefer bullet points and avoid unncessary explanations.
            """

    return call_vllm(prompt, max_tokens=700, temperature=0.3)


def generate_x_post(summary: str) -> str:
    with open("post-examples.json", "r", encoding="utf-8") as f:
        examples = json.load(f)

    examples_str = ""
    for i, example in enumerate(examples, 1):
        examples_str += f"""
                            <example-{i}>
                                    <topic>
                                    {example['topic']}
                                    </topic>

                                    <generated-post>
                                    {example['post']}
                            </generated-post>
                            </example-{i}>
                        """

    prompt = f"""
                You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

                Your task is to generate a post based on a short text summary.
                Your post must be concise and impactful.
                Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

                Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

                Here's the text summary which you should use to generate the post:
                <summary>
                {summary}
                </summary>

                Here are some examples of topics and generated posts:
                <examples>
                {examples_str}
                </examples>

                Please use the tone, language, structure , and style of the examples provided above to generate a post that is engaging and relevant to the topic provided by the user.
                Don't use the content from the examples!
            """

    return call_vllm(prompt, max_tokens=350, temperature=0.7)


def main():
    website_url = input("Website URL: ")
    print("Fetching website HTML...")
    try:
        html_content = get_website_html(website_url)
    except Exception as e:
        print(f"An error occurred while fetching the website: {e}")
        return

    if not html_content:
        print("Failed to fetch the website content. Exiting.")
        return

    print("---------")
    print("Extracting core content from the website...")
    core_content = extract_core_website_content(html_content)
    print("Extracted core content:")
    print(core_content)

    print("---------")
    print("Summarizing the core content...")
    summary = summarize_content(core_content)
    print("Generated summary:")
    print(summary)

    print("---------")
    print("Generating X post based on the summary...")
    x_post = generate_x_post(summary)
    print("Generated X post:")
    print(x_post)


if __name__ == "__main__":
    main()